In [ ]:
import datetime as dt
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from mc_postgres_db.models import (
    Asset,
    AssetType,
    Provider,
    ProviderAsset,
)
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

load_dotenv()

CMC_PRO_API_KEY = os.getenv("CMC_PRO_API_KEY")
POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)


url = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
headers = {"Accept": "application/json", "X-CMC_PRO_API_KEY": CMC_PRO_API_KEY}
response = requests.get(url, headers=headers)
data = response.json()["data"]

In [ ]:
market_cap_df = pd.DataFrame(
    [
        {
            "symbol": item["symbol"],
            "name": item["name"],
            "market_cap": item["quote"]["USD"]["market_cap"],
        }
        for item in data
    ]
)
market_cap_df.sort_values(by="market_cap", ascending=False)

In [ ]:
import requests


def get_asset_code(name: str) -> str:
    try:
        time.sleep(1.0)
        json = requests.get(
            f"https://api.kraken.com/0/public/Assets?asset={name}"
        ).json()
        altname = list(json["result"].values())[0]["altname"]
        return list(
            requests.get(f"https://api.kraken.com/0/public/Assets?asset={altname}")
            .json()["result"]
            .keys()
        )[0]
    except Exception as e:
        print(f"Error getting asset code for {name}: {e}")
        return None


get_asset_code("ETH")

In [ ]:
market_cap_df["asset_code"] = market_cap_df["symbol"].apply(get_asset_code)
market_cap_df.sort_values(by="market_cap", ascending=False, inplace=True)
market_cap_df

In [ ]:
market_cap_kraken_df = market_cap_df.loc[
    ~market_cap_df["asset_code"].isnull()
].reset_index(drop=True)
market_cap_kraken_df.rename(
    columns={"symbol": "name", "name": "description"}, inplace=True
)
market_cap_kraken_df.drop(columns=["market_cap"], inplace=True)
market_cap_kraken_df

In [ ]:
asset_code_to_id_dict = {}
with Session(engine) as session:
    kraken_stmt = select(Provider).where(Provider.name == "Kraken")
    kraken = session.execute(kraken_stmt).scalar_one()

    assets = kraken.get_all_assets(engine)

    for asset in assets:
        if asset.asset_id != 2:
            asset_code_to_id_dict[asset.asset_code] = asset.asset_id

asset_code_to_id_dict

In [ ]:
with Session(engine) as session:
    crypto_asset_type_stmt = select(AssetType).where(
        AssetType.name == "DIGITAL_CURRENCY"
    )
    crypto_asset_type = session.execute(crypto_asset_type_stmt).scalar_one()
    display(crypto_asset_type)

In [ ]:
market_cap_kraken_df["asset_id"] = market_cap_kraken_df["asset_code"].apply(
    lambda x: asset_code_to_id_dict.get(x, None)
)
market_cap_kraken_df["is_active"] = True
market_cap_kraken_df["asset_type"] = crypto_asset_type.id
market_cap_kraken_df

In [ ]:
matched = market_cap_kraken_df.loc[~market_cap_kraken_df["asset_id"].isnull()]
print(f"Matched: {len(matched)}")
print(f"Existing: {len(asset_code_to_id_dict)}")
matched

In [ ]:
unmatched = market_cap_kraken_df.loc[market_cap_kraken_df["asset_id"].isnull()]
batch = unmatched.head(10)
batch

In [ ]:
with Session(engine) as session:
    new_assets = []
    for index, row in unmatched.iterrows():
        new_assets.append(
            Asset(
                name=row["name"],
                asset_type_id=crypto_asset_type.id,
                description=row["description"],
                is_active=row["is_active"],
            )
        )
    session.add_all(new_assets)
    session.commit()

    for asset in new_assets:
        session.refresh(asset)
        print(asset.id)

In [ ]:
asset_name_to_id_dict = {}
for asset in new_assets:
    asset_name_to_id_dict[asset.name] = asset.id

asset_name_to_id_dict

In [ ]:
unmatched["asset_id"] = unmatched["name"].apply(
    lambda x: asset_name_to_id_dict.get(x, None)
)
unmatched

In [ ]:
unmatched.loc[unmatched["asset_id"].isnull()]

In [ ]:
with Session(engine) as session:
    new_provider_assets = []
    for index, row in unmatched.iterrows():
        new_provider_assets.append(
            ProviderAsset(
                date=dt.date(2025, 1, 1),
                provider_id=kraken.id,
                asset_id=row["asset_id"],
                asset_code=row["asset_code"],
                is_active=row["is_active"],
            )
        )
    session.add_all(new_provider_assets)
    session.commit()

    for provider_asset in new_provider_assets:
        session.refresh(provider_asset)
        print(provider_asset)